In [5]:
import pandas as pd
import numpy as np

fy2009 = "/Users/linhchung/Documents/GitHub/Cleaned-OFLC/data/raw/Icert_ LCA_ FY2009.xlsx"

df = pd.read_excel(fy2009)

print(df.shape)
print(df.head())

(87988, 35)
      LCA_CASE_NUMBER     STATUS LCA_CASE_SUBMIT Decision_Date VISA_CLASS  \
0  I-200-09105-016987     DENIED      2009-05-07    2009-05-13       H-1B   
1  I-200-09105-052111  CERTIFIED      2009-04-15    2009-05-27       H-1B   
2  I-200-09105-084817  CERTIFIED      2009-04-15    2009-05-27       H-1B   
3  I-200-09105-089121  WITHDRAWN      2009-04-20    2009-06-18       H-1B   
4  I-200-09105-089121  WITHDRAWN      2009-04-20    2009-06-18       H-1B   

  LCA_CASE_EMPLOYMENT_START_DATE LCA_CASE_EMPLOYMENT_END_DATE  \
0                     2009-07-01                   2012-07-01   
1                     2009-10-01                   2012-09-30   
2                     2009-08-01                   2012-08-01   
3                     2009-05-15                   2009-07-14   
4                     2009-05-15                   2009-07-14   

     LCA_CASE_EMPLOYER_NAME        LCA_CASE_EMPLOYER_ADDRESS  \
0         QUANTA TECHNOLOGY  4020 WESTCHASE BLVD., SUITE 300   
1  DEA

In [7]:
print(df.columns.tolist())

['LCA_CASE_NUMBER', 'STATUS', 'LCA_CASE_SUBMIT', 'Decision_Date', 'VISA_CLASS', 'LCA_CASE_EMPLOYMENT_START_DATE', 'LCA_CASE_EMPLOYMENT_END_DATE', 'LCA_CASE_EMPLOYER_NAME', 'LCA_CASE_EMPLOYER_ADDRESS', 'LCA_CASE_EMPLOYER_CITY', 'LCA_CASE_EMPLOYER_STATE', 'LCA_CASE_EMPLOYER_POSTAL_CODE', 'LCA_CASE_SOC_CODE', 'LCA_CASE_SOC_NAME', 'LCA_CASE_JOB_TITLE', 'LCA_CASE_WAGE_RATE_FROM', 'LCA_CASE_WAGE_RATE_TO', 'LCA_CASE_WAGE_RATE_UNIT', 'FULL_TIME_POS', 'TOTAL_WORKERS', 'LCA_CASE_WORKLOC1_CITY', 'LCA_CASE_WORKLOC1_STATE', 'PW_1', 'PW_UNIT_1', 'PW_SOURCE_1', 'OTHER_WAGE_SOURCE_1', 'YR_SOURCE_PUB_1', 'LCA_CASE_WORKLOC2_CITY', 'LCA_CASE_WORKLOC2_STATE', 'PW_2', 'PW_UNIT_2', 'PW_SOURCE_2', 'OTHER_WAGE_SOURCE_2', 'YR_SOURCE_PUB_2', 'LCA_CASE_NAICS_CODE']


In [9]:
import pandas as pd

df = pd.read_excel(fy2009)

df["LCA_CASE_EMPLOYER_NAME"] = (
    df["LCA_CASE_EMPLOYER_NAME"]
    .astype("string")
    .str.strip()
    .str.upper()
)

keywords = [
    "NVIDIA",
    "APPLE",
    "MICROSOFT",
    "MICRON",
    "ADVANCED MICRO DEVICES",
    "AMAZON",
    "META",
    "FACEBOOK",
    "ALPHABET",
    "GOOGLE",
    "TESLA"
]

mask = df["LCA_CASE_EMPLOYER_NAME"].str.contains(
    "|".join(keywords),
    na=False,
    regex=True
)

print(
    df.loc[mask, "LCA_CASE_EMPLOYER_NAME"]
    .value_counts()
    .to_string()
)

LCA_CASE_EMPLOYER_NAME
MICROSOFT CORPORATION                                 791
GOOGLE INC.                                           182
MICRON TECHNOLOGY, INC.                               123
AMAZON GLOBAL RESOURCES, INC.                         100
APPLE INC.                                             57
ADVANCED MICRO DEVICES, INC.                           54
FACEBOOK, INC.                                         30
NVIDIA CORPORATION                                     29
TESLA MOTORS, INC.                                      9
APPLE EDUCATIONAL SERVICES, INC.                        7
APPLE INC                                               6
METASENSE, INC.                                         6
CENTRAL METAL, INC.                                     6
AMAZON FULFILLMENT SERVICES, INC.                       4
IBM CORPORATION C/O JEAN APPLEMAN                       4
CUSTOM METAL OF PECONIC, INC.                           3
METAPRO USA,LLC                                  

In [11]:
company_map = {
    "NVIDIA CORPORATION": "NVIDIA",
    "APPLE INC.": "Apple",
    "MICROSOFT CORPORATION": "Microsoft",
    "MICRON TECHNOLOGY, INC.": "Micron",
    "ADVANCED MICRO DEVICES, INC.": "AMD",
    "AMAZON.COM, INC.": "Amazon",
    "AMAZON.COM SERVICES LLC": "Amazon",
    "META PLATFORMS, INC.": "Meta",
    "FACEBOOK, INC.": "Meta",
    "ALPHABET INC.": "Alphabet",
    "GOOGLE LLC": "Alphabet",
    "TESLA, INC.": "Tesla",
    "TESLA MOTORS, INC.": "Tesla"
}

df["Company"] = df["LCA_CASE_EMPLOYER_NAME"].map(company_map)

df = df[df["Company"].notna()].copy()

print(df["Company"].value_counts())

Company
Microsoft    791
Micron       123
Apple         57
AMD           54
Meta          30
NVIDIA        29
Tesla          9
Name: count, dtype: int64


In [13]:
df_clean = df.copy()

In [15]:
city_names = {
    "SEATTLE": "Seattle",
    "SAN JOSE": "San Jose",
    "NEW YORK": "New York",
    "AUSTIN": "Austin",
    "DALLAS": "Dallas"
}

# Standardize city names
df["LCA_CASE_WORKLOC1_CITY"] = (
    df["LCA_CASE_WORKLOC1_CITY"]
    .astype("string")
    .str.strip()
    .str.upper()
)

# Create the City column
df["City"] = df["LCA_CASE_WORKLOC1_CITY"].map(city_names)

# Keep only the five selected cities
df = df[df["City"].notna()].copy()

print(df["City"].value_counts())

City
Austin      27
San Jose     7
New York     5
Seattle      4
Name: count, dtype: int64


In [17]:
print(df["LCA_CASE_WAGE_RATE_UNIT"].value_counts())

LCA_CASE_WAGE_RATE_UNIT
Year    43
Name: count, dtype: int64


In [19]:
import pandas as pd

# Convert wage values to numbers
df["LCA_CASE_WAGE_RATE_FROM"] = pd.to_numeric(
    df["LCA_CASE_WAGE_RATE_FROM"],
    errors="coerce"
)

# Standardize wage units
df["LCA_CASE_WAGE_RATE_UNIT"] = (
    df["LCA_CASE_WAGE_RATE_UNIT"]
    .astype("string")
    .str.strip()
    .str.upper()
)

# Conversion factors
wage_multiplier = {
    "YEAR": 1,
    "MONTH": 12,
    "BI-WEEKLY": 26,
    "WEEK": 52,
    "HOUR": 2080
}

# Create annual wage column
df["Annual_Wage"] = (
    df["LCA_CASE_WAGE_RATE_FROM"]
    * df["LCA_CASE_WAGE_RATE_UNIT"].map(wage_multiplier)
)

# Check results
print(df[[
    "Company",
    "City",
    "LCA_CASE_WAGE_RATE_FROM",
    "LCA_CASE_WAGE_RATE_UNIT",
    "Annual_Wage"
]].head(10))

     Company      City  LCA_CASE_WAGE_RATE_FROM LCA_CASE_WAGE_RATE_UNIT  \
1243     AMD    Austin                  88707.0                    YEAR   
1323     AMD    Austin                  63357.0                    YEAR   
1446  Micron  San Jose                 108000.0                    YEAR   
1533  Micron  San Jose                 126351.0                    YEAR   
1534  Micron  San Jose                 126351.0                    YEAR   
2096     AMD    Austin                  70330.0                    YEAR   
2311  Micron  San Jose                  95950.0                    YEAR   
2312  Micron  San Jose                  95950.0                    YEAR   
3398     AMD    Austin                  70330.0                    YEAR   
3890     AMD    Austin                  63357.0                    YEAR   

      Annual_Wage  
1243      88707.0  
1323      63357.0  
1446     108000.0  
1533     126351.0  
1534     126351.0  
2096      70330.0  
2311      95950.0  
2312      9595

In [21]:
print("Missing wages:", df["Annual_Wage"].isna().sum())

print(df["Annual_Wage"].describe())

Missing wages: 0
count        43.000000
mean     105081.023256
std       73649.723681
min       63357.000000
25%       75867.000000
50%       84864.000000
75%      102325.000000
max      525000.000000
Name: Annual_Wage, dtype: float64


In [23]:
df = df.dropna(subset=["Annual_Wage"]).copy()

df = df[df["Annual_Wage"] > 0].copy()

In [25]:
wage_summary = (
    df
    .groupby(["City", "Company"])
    .agg(
        Number_of_Records=("Annual_Wage", "count"),
        Median_Wage=("Annual_Wage", "median"),
        Mean_Wage=("Annual_Wage", "mean")
    )
    .reset_index()
)

print(wage_summary)

       City    Company  Number_of_Records  Median_Wage      Mean_Wage
0    Austin        AMD                 26      81120.0  106118.923077
1    Austin     NVIDIA                  1      81700.0   81700.000000
2  New York  Microsoft                  5      84623.0   89543.200000
3  San Jose     Micron                  7     108000.0  109178.857143
4   Seattle  Microsoft                  4     110000.0  116431.000000


In [27]:
columns_to_keep = [
    "Company",
    "LCA_CASE_EMPLOYER_NAME",
    "LCA_CASE_JOB_TITLE",
    "LCA_CASE_SOC_NAME",
    "City",
    "LCA_CASE_WAGE_RATE_FROM",
    "LCA_CASE_WAGE_RATE_UNIT",
    "Annual_Wage"
]

df_clean = df[columns_to_keep].copy()

df_clean.to_csv(
    "OFLC_BigTech_Wages_2009.csv",
    index=False
)